# RAG Application for alab-mart ecommerce platform

### Data Loading

In [1]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(
    "data/Alab.pdf"
)

documents = loader.load()
print(len(documents))

C:\Users\DELL\AppData\Local\Temp\ipykernel_11496\2244680564.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
d:\alab\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


36


Data Splitting 

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

chunks = text_splitter.split_documents(documents)
print(len(chunks))

97


In [3]:
print(chunks[0].page_content)

Alab-mart 
 
Alab-mart is a ecommerce company which is completely working online through a ecommerce 
website we are partnered with different book stores and hardware companies to sell the books 
,hardware devices related to AI. Basically in this platform only ai related stuff will be available like 
books,hardware,ai assistants . 
 
The owner of this ecommerce platform is A.Kalyan Sai,He is an ai engineer with 2 internship 
experiences one at kodemelon technologies and other one is at celtm there he learnt about cutting 
edge ai technologies like rag ,computer vision, automation pipelines. 
In this mart the offers will be on 3 days per year  
that is on 09-09,26-11 and 3-11 of every year 
If you order any item or bluck of items then all the items will be delivered within 2 days of your 
purchase  
For any queries contact details are as follows 
for ecommerce website queries  
contact no: 1234567890 
email: ecommerce@alab-mart.com 
for product enquires  
contact no : 9876543210


Create Embeddings

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2375.93it/s]


Create Vectorstore 

In [5]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory="rag/chroma_db",
)


Create Retrieval

In [6]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 10,
        "fetch_k": 20,
        "lambda_mult": 0.5
    }
)

In [7]:
query = "Tell me about alab-mart"

retrieved_docs = retriever.invoke(query)
print(retrieved_docs[0].page_content)

Alab-mart 
 
Alab-mart is a ecommerce company which is completely working online through a ecommerce 
website we are partnered with different book stores and hardware companies to sell the books 
,hardware devices related to AI. Basically in this platform only ai related stuff will be available like 
books,hardware,ai assistants . 
 
The owner of this ecommerce platform is A.Kalyan Sai,He is an ai engineer with 2 internship 
experiences one at kodemelon technologies and other one is at celtm there he learnt about cutting 
edge ai technologies like rag ,computer vision, automation pipelines. 
In this mart the offers will be on 3 days per year  
that is on 09-09,26-11 and 3-11 of every year 
If you order any item or bluck of items then all the items will be delivered within 2 days of your 
purchase  
For any queries contact details are as follows 
for ecommerce website queries  
contact no: 1234567890 
email: ecommerce@alab-mart.com 
for product enquires  
contact no : 9876543210


In [8]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def rerank_documents(query, documents, top_k=5):
    pairs = [
        (query, doc.page_content)
        for doc in documents
    ]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(scores, documents),
        key=lambda x: x[0],
        reverse=True
    )

    return [
        doc
        for score, doc in ranked[:top_k]
    ]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2445.18it/s]


In [9]:
def format_docs(docs):
    return "\n\n".join(
        [
            f"Document {i+1}\n{doc.page_content}"
            for i, doc in enumerate(docs)
        ]
    )

Connect LLM

In [10]:
from langchain_ollama import ChatOllama

llm  = ChatOllama(
    model = "qwen2.5:3b",
    temperature = 0
)

In [11]:
llm.invoke("Tell me about alab-mart")

AIMessage(content='I\'m sorry for the confusion, but I couldn\'t find any specific information on "alab-mart" related to Alibaba Cloud or any other well-known companies. It\'s possible that this term might be misspelled or used in a context not widely known.\n\nIf you could provide more details about where you encountered this term (e.g., from which website, document, or conversation), I would be better equipped to help you understand what "alab-mart" refers to. Alternatively, if you have another topic you\'d like to know more about, feel free to ask and I\'ll do my best to assist you.', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-07-27T14:13:09.45935Z', 'done': True, 'done_reason': 'stop', 'total_duration': 49327892400, 'load_duration': 21102448900, 'prompt_eval_count': 36, 'prompt_eval_duration': 2161507000, 'eval_count': 127, 'eval_duration': 25935083000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--01

ChatPromptTemplate 

In [12]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are ALAB-MART AI Assistant, a professional virtual assistant for ALAB-MART, an AI-focused e-commerce platform.

Your responsibilities:
- Help customers understand products, policies, and services.
- Answer ONLY using the provided context.
- Never make up information.
- If the answer is not present in the context, politely say:
  "I'm sorry, but I couldn't find that information in the ALAB-MART knowledge base."
- Do not mention the context, embeddings, vector database, retrieval process, or AI models.
- Be polite, professional, and concise.
- If multiple products satisfy the user's request, compare them in a clear table.
- If the user asks for recommendations, explain why each recommendation is suitable.
- Preserve technical specifications exactly as provided.
- When answering policy questions (shipping, refund, warranty, returns, payment), provide a concise summary.
- If the question is ambiguous, ask one clarifying question before answering.
- Format the response using Markdown when appropriate.

Context:
{context}

Customer Question:
{question}

Response:
""")

In [13]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda


def retrieve_and_rerank(query):
    docs = retriever.invoke(query)
    docs = rerank_documents(query, docs)   # Your reranking function
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {
        "context": RunnableLambda(retrieve_and_rerank),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [14]:
response = chain.invoke(
    "Tell me about alab-mart."
)

print(response)

Alab-mart is an online e-commerce platform dedicated exclusively to Artificial Intelligence (AI) products. Our mission is to provide students, developers, researchers, and AI professionals with a trusted marketplace where they can purchase AI books, AI hardware devices, and AI-powered smart assistant devices.

We partner with leading publishers, electronics manufacturers, and technology companies to ensure customers receive genuine products at competitive prices. Alab-mart offers three different product categories:

1. **AI Books**: We carry 10 books including "Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow" by O'Reilly Media.
2. **AI Hardware Devices**: Products like the BeagleBone AI-64 development board from Arduino.
3. **AI Assistant Devices**: Examples include smart assistant devices.

Alab-mart hosts 3 special offers each year on September 9th, November 26th, and November 3rd. Orders placed within this timeframe will be delivered within two days of purchase.

